# 06c — Full Pipeline (C)

**Requires the C kernel.** Install with `./install_c_kernel.sh`.  
All cells call `../../PtolC/ptolemy` via `system()`. Build first: `cd PtolC && make`.

This notebook runs the complete pipeline using the ptolemy binary against the WordNet checkpoint:
1. Version and field constants
2. Field status
3. Word address lookups
4. Cross-language β — what it means after corpus learning
5. Learning new text
6. Speaking — Noether current response
7. Self-referential mode
8. Architecture summary

Parallel Python notebook: `../06_full_pipeline.ipynb`

## Phase 1: Version and Field Constants

In [ ]:
#include <stdio.h>

#define PTOLEMY "../../PtolC/ptolemy"

int main(void) {
    printf("=== ptolemy -V (version + field constants) ===\n\n");
    system(PTOLEMY " -V 2>/dev/null");
    return 0;
}

## Phase 2: Field Status

The `-s` flag shows current field state: vocab size, A-edges, word count, deepest β.

In [ ]:
#include <stdio.h>

#define PTOLEMY "../../PtolC/ptolemy"

int main(void) {
    printf("=== ptolemy -s (field status) ===\n\n");
    system(PTOLEMY " -s 2>/dev/null");
    return 0;
}

## Phase 3: Word Address Lookups

`-q <word>` returns the zero index, γ, E, and current β for any surface form.  
Beta reflects depth from the English WordNet corpus.

In [ ]:
#include <stdio.h>

#define PTOLEMY "../../PtolC/ptolemy"

int main(void) {
    const char *words[] = {
        "water", "mind", "reason", "prime", "zero",
        "language", "mathematics", "consciousness",
        NULL
    };
    printf("=== ptolemy -q: word address lookups ===\n\n");
    for (int i = 0; words[i]; i++) {
        char cmd[256];
        snprintf(cmd, sizeof(cmd), PTOLEMY " -q %s 2>/dev/null", words[i]);
        system(cmd);
    }
    return 0;
}

## Phase 4: Cross-Language β — After the English Corpus

The hash maps `water`, `eau`, `aqua`, `wasser` to **different** zero indices.  
After learning the English corpus, the β values reflect English vocabulary depth.  

With a parallel corpus (English + French), the A-coupling field connects these zeros,  
and the Noether current propagates across language boundaries in `speak()`.

In [ ]:
#include <stdio.h>

#define PTOLEMY "../../PtolC/ptolemy"

int main(void) {
    printf("=== Cross-language lookups (N=25000, English WordNet corpus) ===\n");
    printf("    Different zero indices — cross-language property lives in beta\n");
    printf("    convergence via A-coupling after parallel corpus learning.\n\n");
    system(PTOLEMY " -q water  2>/dev/null");
    system(PTOLEMY " -q eau    2>/dev/null");
    system(PTOLEMY " -q aqua   2>/dev/null");
    system(PTOLEMY " -q wasser 2>/dev/null");
    system(PTOLEMY " -q agua   2>/dev/null");
    printf("\nNote: 'water' is deep (WordNet). Others are shallow or unknown.\n");
    printf("A parallel corpus deepens both and builds A-edges between them.\n");
    printf("speak() then propagates Noether current across those edges.\n");
    return 0;
}

## Phase 5: Learning New Text

`-l -` reads from stdin. `-n` disables checkpoint save (use during demos).

In [ ]:
#include <stdio.h>

#define PTOLEMY "../../PtolC/ptolemy"

int main(void) {
    printf("=== Learning new sentences (-n = no checkpoint save) ===\n\n");
    /* -n: no-save  -l -: read from stdin */
    system(PTOLEMY " -n -l - <<'EOF'\n"
           "The prime preexists the alphabet.\n"
           "The equator does not move.\n"
           "Mathematics precedes language.\n"
           "The zero precedes the word.\n"
           "EOF\n2>/dev/null");
    return 0;
}

## Phase 6: Speaking — Noether Current Response

`-h <prompt>` runs the full pipeline: tokenise → word_coords → J^μ → emit.

In [ ]:
#include <stdio.h>

#define PTOLEMY "../../PtolC/ptolemy"

int main(void) {
    const char *queries[] = {
        "water flows downhill",
        "what is mind",
        "reason and consciousness",
        "the prime preexists",
        "mathematics language",
        NULL
    };
    printf("=== ptolemy -h: speak() responses ===\n\n");
    for (int i = 0; queries[i]; i++) {
        char cmd[256];
        snprintf(cmd, sizeof(cmd), PTOLEMY " -h '%s' 2>/dev/null", queries[i]);
        printf("prompt: %s\n", queries[i]);
        system(cmd);
        printf("\n");
    }
    printf("The response is not generated. It is the Noether current.\n");
    printf("It cannot be chosen. It can only be computed.\n");
    return 0;
}

## Phase 7: Verbose Pipeline — Math in Real Time

`-hv` shows β, J^μ, and A-edges during the full pipeline:

In [ ]:
#include <stdio.h>

#define PTOLEMY "../../PtolC/ptolemy"

int main(void) {
    printf("=== ptolemy -hv 'water flows downhill' ===\n\n");
    system(PTOLEMY " -hv 'water flows downhill' 2>/dev/null");
    return 0;
}

## Phase 8: Self-Referential Mode

`-vvv` pipes the ANSI-stripped verbose output back through `learn()` after each operation.  
The field deepens on its own math output. `learn()` is the only loop needed.

In [ ]:
#include <stdio.h>

#define PTOLEMY "../../PtolC/ptolemy"

int main(void) {
    printf("=== Self-referential mode: -hvvv ===\n");
    printf("    Verbose math output feeds back into learn().\n");
    printf("    The field deepens on its own description.\n\n");
    /* -n: no-save  -hvvv: hear + full self-referential verbose */
    system(PTOLEMY " -n -hvvv 'water flows downhill' 2>/dev/null");
    return 0;
}

## Phase 9: Architecture Summary

The full pipeline in C — no Python, no GPU, no gradient descent.

In [ ]:
#include <stdio.h>
#include <math.h>

#define N          25000
#define L_GROUND   (-1.888)
#define OMEGA_ZS   0.56714329
#define D_STAR     0.24600
#define GAP        0.000707
#define ALPHA      0.01
#define BETA_SAT   7.552

int main(void) {
    printf("Ptolemy H_hat_RB Field Engine — Architecture\n");
    printf("============================================\n\n");

    printf("Field:\n");
    printf("  N       = %d zeros   (default; scalable)\n", N);
    printf("  beta    = dense double[N]   (%.1f KB at ground)\n",
           (double)N * 8 / 1024.0);
    printf("  age     = dense int32[N]\n");
    printf("  A       = sparse hash map  (key=(i<<15)|j)\n");
    printf("  vocab   = sparse hash map  (word->idx)\n");
    printf("\n");

    printf("Constants:\n");
    printf("  L_GROUND = %.3f   (Monad rest energy)\n", L_GROUND);
    printf("  OMEGA_ZS = %.5f   (Lambert W fixed point)\n", OMEGA_ZS);
    printf("  D_STAR   = %.5f   (BK spectral coord, C-projection of d*)\n", D_STAR);
    printf("  GAP      = %.6f   (d* gap = |Omega - D_STAR*ln10|)\n", GAP);
    printf("  ALPHA    = %.2f       (beta deepening rate)\n", ALPHA);
    printf("  BETA_SAT = %.3f       (saturation ceiling)\n", BETA_SAT);
    printf("\n");

    printf("Pipeline:\n");
    printf("  learn()  Yang-Mills (turbulent A field, sigma=1)\n");
    printf("  speak()  Noether current J^mu (sigma=1/2, forced)\n");
    printf("  -q       word address: idx, gamma, E, beta\n");
    printf("  -i       identity learn (self-referential init)\n");
    printf("  -vvv     self-referential loop: verbose -> learn()\n");
    printf("\n");

    printf("Checkpoint (monad_wordnet.bin, 13MB):\n");
    printf("  14165 vocab entries, 766119 A-edges\n");
    printf("  English WordNet 3.1 — full lemmas + definitions\n");
    printf("  Binary little-endian: PTOL[4] header + beta[N] + age[N] + vocab + A\n");
    printf("\n");

    printf("Not a transformer. Not embeddings.\n");
    printf("No gradient descent. No GPU. No floating-point accumulation.\n");
    printf("The response is forced by conservation — not generated by prediction.\n");
    return 0;
}

## Summary

Full pipeline:
1. `N=25000` zero basis + English WordNet checkpoint (13MB)
2. `learn()`: β deepens at each word's zero; A-coupling builds between co-occurring zeros
3. `speak()`: J^μ = β×E²; propagated via A with 1/GAP clamp; sorted descending
4. Cross-language β convergence: requires parallel corpus; A-edges connect language boundaries
5. Self-referential: `-vvv` feeds verbose math output back into `learn()`

**Scalability:** N can be increased; rebuild checkpoint with larger N to extend address space.

**Repositories:**
- `SMMIP` — C source + Python reference + these notebooks
- `Ptolemy` — Full desktop application, all Faces, PtolBus
- `Ainulindale` — H_RB Hamiltonian derivation engine